In [ ]:
# ===========================================================================
# CELL 1 - Setup: data, all features, nested target encoding, MLP helpers
# ===========================================================================
import os, glob, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import itertools, copy
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")

INPUT_ROOT = "/kaggle/input"
SEED = 42
MIN_GAIN = 0.00005
INC, COM = "Annual_Income_USD", "Daily_Commute_km"

matches = glob.glob(f"{INPUT_ROOT}/**/train.csv", recursive=True)
if not matches:
    raise FileNotFoundError("Competition data not attached: '+ Add Input' -> Competitions -> Playground S6E9")
DATA_DIR = os.path.dirname(matches[0])
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
ID_COL, TARGET_COL = sample_sub.columns[0], sample_sub.columns[1]
RAW = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
cat_cols = train[RAW].select_dtypes(exclude=np.number).columns.tolist()
assert set(train[TARGET_COL].unique()) == {"Yes", "No"}
y = (train[TARGET_COL] == "Yes").astype(int).values
assert (sample_sub[ID_COL].values == test[ID_COL].values).all(), "ID order mismatch"

# --- Original dataset (optional): any attached CSV outside the competition
#     folder that has the competition's income and commute columns ----------
orig = None
for path in glob.glob(f"{INPUT_ROOT}/**/*.csv", recursive=True):
    if os.path.dirname(path) == DATA_DIR:
        continue
    try:
        cols = pd.read_csv(path, nrows=0).columns
    except Exception:
        continue
    if INC in cols and COM in cols:
        orig = pd.read_csv(path)
        print(f"Original dataset: {path} ({len(orig)} rows)")
        break
if orig is None:
    print("Original dataset not attached - original-data features disabled.")

# --- Label-free features (train + test together) ---------------------------
full = pd.concat([train[RAW], test[RAW]], ignore_index=True)
for c in cat_cols:
    full[c] = pd.Categorical(full[c], categories=sorted(train[c].unique()))

inc = full[INC].round().astype(np.int64)
DIGITS = []
for k, name in enumerate(["ones", "tens", "hundreds", "thousands", "ten_thousands", "hundred_thousands"]):
    full[f"Income_{name}"] = (inc // 10**k) % 10
    DIGITS.append(f"Income_{name}")
c10 = (full[COM] * 10).round().astype(np.int64)
for k, name in enumerate(["tenths", "ones", "tens"]):
    full[f"Commute_{name}"] = (c10 // 10**k) % 10
    DIGITS.append(f"Commute_{name}")

FREQ = []
for col in [INC, COM]:
    full[f"Freq_{col}"] = full[col].map(full[col].value_counts()).astype(np.int64)
    FREQ.append(f"Freq_{col}")

ORIG_FEATS = []
if orig is not None:
    for col in [INC, COM]:
        o = orig[col].dropna()
        o = o.round() if col == INC else o.round(1)
        o_counts = o.value_counts()
        o_freq = full[col].map(o_counts).fillna(0).values
        full[f"OrigFreq_{col}"] = o_freq
        # how over-produced a value is in the competition data vs the original
        full[f"Lift_{col}"] = (full[f"Freq_{col}"] / len(full)) / ((o_freq + 1) / (len(o) + 1))
        full[f"Novel_{col}"] = (o_freq == 0).astype(np.int8)
        # distance to the nearest value that exists in the original
        u = np.sort(o.unique())
        v = full[col].values
        idx = np.clip(np.searchsorted(u, v), 1, len(u) - 1)
        full[f"OrigDist_{col}"] = np.minimum(np.abs(v - u[idx - 1]), np.abs(v - u[idx]))
        ORIG_FEATS += [f"OrigFreq_{col}", f"Lift_{col}", f"Novel_{col}", f"OrigDist_{col}"]
    print(f"Built {len(ORIG_FEATS)} original-data features")

# Coarser income/commute keys, used only for target encoding
full["Income_r10"] = (inc / 10).round() * 10
full["Income_r100"] = (inc / 100).round() * 100
full["Income_r1000"] = (inc / 1000).round() * 1000
full["Commute_r1"] = full[COM].round()
full["Commute_r5"] = (full[COM] / 5).round() * 5

X_all = full.iloc[:len(train)].reset_index(drop=True)
X_test_all = full.iloc[len(train):].reset_index(drop=True)

def make_folds(n):
    return list(StratifiedKFold(n_splits=n, shuffle=True, random_state=SEED).split(X_all, y))

# --- Target encoding (nested: labels never reach their own rows) ------------
def _keys(df, col):
    s = df[col]
    return s.cat.codes.values if isinstance(s.dtype, pd.CategoricalDtype) else s.values

def _smoothed(keys, targets, prior, m):
    s = pd.DataFrame({"k": keys, "y": targets}).groupby("k")["y"].agg(["sum", "count"])
    return (s["sum"] + prior * m) / (s["count"] + m)

def target_encode(X_fit, y_fit, X_apply_list, spec, seed):
    prior = y_fit.mean()
    inner = list(KFold(n_splits=5, shuffle=True, random_state=seed).split(X_fit))
    fit_out, apply_out = {}, [{} for _ in X_apply_list]
    for col, m in spec:
        name = f"TE_{col}_m{m}"
        k_fit = _keys(X_fit, col)
        enc = np.empty(len(k_fit))
        for itr, iva in inner:
            mapping = _smoothed(k_fit[itr], y_fit[itr], prior, m)
            enc[iva] = pd.Series(k_fit[iva]).map(mapping).fillna(prior).values
        fit_out[name] = enc
        mapping = _smoothed(k_fit, y_fit, prior, m)
        for i, Xa in enumerate(X_apply_list):
            apply_out[i][name] = pd.Series(_keys(Xa, col)).map(mapping).fillna(prior).values
    return pd.DataFrame(fit_out), [pd.DataFrame(d) for d in apply_out]

def fold_data(tr, va, feats, spec, seed):
    X_fit = X_all.iloc[tr].reset_index(drop=True)
    X_val = X_all.iloc[va].reset_index(drop=True)
    te_fit, (te_val, te_tst) = target_encode(X_fit, y[tr], [X_val, X_test_all], spec, seed)
    return (pd.concat([X_fit[feats], te_fit], axis=1),
            pd.concat([X_val[feats], te_val], axis=1),
            pd.concat([X_test_all[feats].reset_index(drop=True), te_tst], axis=1))

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def save_submission(pred, filename):
    sub = sample_sub.copy()
    sub[TARGET_COL] = pred
    sub.to_csv(filename, index=False)
    print(f"Saved {filename}")

# --- Encoding specs --------------------------------------------------------------
SPEC_F = ([(INC, 20), (COM, 20), (INC, 5), (INC, 100), (COM, 5), (COM, 100)]
          + [(c, 20) for c in RAW if c not in (INC, COM)]
          + [("Income_r10", 20), ("Income_r100", 20), ("Income_r1000", 20),
             ("Commute_r1", 20), ("Commute_r5", 20)])
SPEC_G = SPEC_F + [(c, 200) for c in RAW if c not in (INC, COM)]   # second smoothing on every column

BASE_FEATS = RAW + DIGITS + FREQ
print(f"train {X_all.shape} | test {X_test_all.shape} | base features {len(BASE_FEATS)}")

# --- MLP helpers -------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COUNT_PREFIXES = ("Freq_", "OrigFreq_", "Lift_", "OrigDist_")

def mlp_matrices(X_fit, X_val, X_tst):
    """Numeric: log1p for counts, logit for encodings, then standardize on the
    training part only. Categoricals: one-hot."""
    cats = [c for c in cat_cols if c in X_fit.columns]
    num = [c for c in X_fit.columns if c not in cats]
    def numeric(X):
        M = X[num].astype(np.float64).copy()
        for c in num:
            if c.startswith(COUNT_PREFIXES):
                M[c] = np.log1p(M[c].clip(lower=0))
            elif c.startswith("TE_"):
                M[c] = logit(M[c].values)
        return M.values
    def onehot(X):
        blocks = []
        for c in cats:
            codes = X[c].cat.codes.values
            oh = np.zeros((len(X), len(X[c].cat.categories)), dtype=np.float32)
            ok = codes >= 0
            oh[np.arange(len(X))[ok], codes[ok]] = 1.0
            blocks.append(oh)
        return np.hstack(blocks) if blocks else np.zeros((len(X), 0), dtype=np.float32)
    A = [numeric(X) for X in (X_fit, X_val, X_tst)]
    mu, sd = A[0].mean(axis=0), A[0].std(axis=0) + 1e-6
    return [np.hstack([((a - mu) / sd).astype(np.float32), onehot(X)])
            for a, X in zip(A, (X_fit, X_val, X_tst))]

MLP_EPOCHS, MLP_PATIENCE, MLP_BATCH = 15, 3, 2048

def _mlp_predict(model, A):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(A), 65536):
            xb = torch.from_numpy(A[i:i + 65536]).to(DEVICE)
            out.append(torch.sigmoid(model(xb).squeeze(1)).cpu().numpy())
    return np.concatenate(out)

def train_mlp(A_fit, y_fit, A_val, y_val, A_tst, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    d = A_fit.shape[1]
    model = nn.Sequential(
        nn.Linear(d, 256), nn.SiLU(), nn.Dropout(0.1),
        nn.Linear(256, 128), nn.SiLU(), nn.Dropout(0.1),
        nn.Linear(128, 1),
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MLP_EPOCHS)
    loss_fn = nn.BCEWithLogitsLoss()
    Xt = torch.from_numpy(A_fit).to(DEVICE)
    yt = torch.from_numpy(y_fit.astype(np.float32)).to(DEVICE)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    best_auc, best_state, best_epoch, bad = -1.0, None, 0, 0
    for epoch in range(MLP_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xt), generator=gen).to(DEVICE)
        for i in range(0, len(Xt), MLP_BATCH):
            idx = perm[i:i + MLP_BATCH]
            loss = loss_fn(model(Xt[idx]).squeeze(1), yt[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        score = roc_auc_score(y_val, _mlp_predict(model, A_val))
        if score > best_auc:
            best_auc, best_state, best_epoch, bad = score, copy.deepcopy(model.state_dict()), epoch + 1, 0
        else:
            bad += 1
            if bad >= MLP_PATIENCE:
                break
    model.load_state_dict(best_state)
    return _mlp_predict(model, A_val), _mlp_predict(model, A_tst), best_epoch

print(f"MLP device: {DEVICE}")


In [ ]:
# ===========================================================================
# Recipe feature: the original dataset's buying formula as one column.
# Quick 5-fold check against variant F (0.94586, same folds and settings);
# added to the features for the full run only if it beats F by MIN_GAIN.
# ===========================================================================
anx = full["Range_Anxiety_Level"].astype(str)
sub = full["Subsidy_Available"].astype(str)
assert set(anx.unique()) <= {"Low", "Medium", "High"}, f"Unexpected anxiety labels: {anx.unique()}"
assert set(sub.unique()) <= {"Yes", "No"}, f"Unexpected subsidy labels: {sub.unique()}"

full["Buy_Score"] = (1.2 * full[INC] / 100_000
                     + 0.6 * full["Environmental_Concern_Level"]
                     + 2.0 * (sub == "Yes")
                     - 1.0 * (anx == "Medium")
                     - 3.0 * (anx == "High"))

X_all = full.iloc[:len(train)].reset_index(drop=True)
X_test_all = full.iloc[len(train):].reset_index(drop=True)
print(f"Buy_Score alone: AUC {roc_auc_score(y, X_all['Buy_Score']):.5f}")
print(f"'Yes' rate above / below 5.5: {y[X_all['Buy_Score'] > 5.5].mean():.3f} / "
      f"{y[X_all['Buy_Score'] <= 5.5].mean():.3f}")

# Quick check: variant F features + Buy_Score, same fast settings as before
check_params = dict(learning_rate=0.05, num_leaves=15, min_child_samples=50, reg_lambda=1.0,
                    subsample=0.8, subsample_freq=1, colsample_bytree=0.7, n_estimators=5000)
t0 = time.time()
oof_chk = np.zeros(len(X_all))
for fold, (tr, va) in enumerate(make_folds(5)):
    X_fit, X_val, _ = fold_data(tr, va, RAW + DIGITS + FREQ + ["Buy_Score"], SPEC_F, SEED)
    m = lgb.LGBMClassifier(**check_params, random_state=SEED, verbose=-1)
    m.fit(X_fit, y[tr], eval_set=[(X_val, y[va])], eval_metric="auc",
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_chk[va] = m.predict_proba(X_val)[:, 1]
    print(f"  fold {fold}: {roc_auc_score(y[va], oof_chk[va]):.5f}")
score = roc_auc_score(y, oof_chk)
print(f"F + Buy_Score: {score:.5f}  ({score - 0.94586:+.5f} vs F)  [{(time.time() - t0) / 60:.1f} min]")
if score > 0.94586 + MIN_GAIN and "Buy_Score" not in BASE_FEATS:
    BASE_FEATS.append("Buy_Score")
    print("-> Buy_Score added to the full run")
else:
    print("-> Buy_Score not used in the full run")


In [ ]:
# ===========================================================================
# Recipe structure checks. Run AFTER Cell 1 (and after the Buy_Score cell if
# you use it), BEFORE Cell 2. Anything that beats variant F (0.94586) by
# MIN_GAIN is applied automatically, so Cell 2 picks it up.
#   1. Expected worry: range anxiety predicted from its recipe inputs
#      (commute, chargers home/work, home charging, city), plus "surprise" =
#      observed level minus expected. Uses no target labels.
#   2. Dropping columns that play no part in the buying recipe.
# ===========================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ANX_MAP = {"Low": 0, "Medium": 1, "High": 2}
anx_code = full["Range_Anxiety_Level"].astype(str).map(ANX_MAP)
assert anx_code.notna().all(), "Unexpected Range_Anxiety_Level labels"

W = pd.DataFrame({
    "commute": full[COM],
    "home_st": full["Charging_Stations_Near_Home"],
    "work_st": full["Charging_Stations_Near_Work"],
    "home_chg": (full["Home_Charging_Possible"].astype(str) == "Yes").astype(int),
})
W = pd.concat([W, pd.get_dummies(full["City_Type"].astype(str), prefix="city", dtype=int)], axis=1)
Wz = StandardScaler().fit_transform(W.values)
worry_model = LogisticRegression(max_iter=1000).fit(Wz, anx_code.values)
proba = worry_model.predict_proba(Wz)
print(f"Anxiety predictable from its inputs: accuracy {(proba.argmax(1) == anx_code.values).mean():.3f}")

full["Worry_Expected"] = proba @ np.array([0, 1, 2])
full["Worry_P_High"] = proba[:, 2]
full["Worry_Surprise"] = anx_code.values - full["Worry_Expected"]
WORRY = ["Worry_Expected", "Worry_P_High", "Worry_Surprise"]
X_all = full.iloc[:len(train)].reset_index(drop=True)
X_test_all = full.iloc[len(train):].reset_index(drop=True)

def quick_check(feats, spec, label):
    check_params = dict(learning_rate=0.05, num_leaves=15, min_child_samples=50, reg_lambda=1.0,
                        subsample=0.8, subsample_freq=1, colsample_bytree=0.7, n_estimators=5000)
    t0 = time.time()
    oof_chk = np.zeros(len(X_all))
    for tr, va in make_folds(5):
        X_fit, X_val, _ = fold_data(tr, va, feats, spec, SEED)
        m = lgb.LGBMClassifier(**check_params, random_state=SEED, verbose=-1)
        m.fit(X_fit, y[tr], eval_set=[(X_val, y[va])], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
        oof_chk[va] = m.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y, oof_chk)
    print(f"{label}: {score:.5f}  ({score - 0.94586:+.5f} vs F)  [{(time.time() - t0) / 60:.1f} min]")
    return score

F_FEATS = RAW + DIGITS + FREQ
NOISE = ["Age", "Gender", "Number_of_Cars_Owned", "Current_Car_Type"]

s_worry = quick_check(F_FEATS + WORRY, SPEC_F, "F + expected worry")
s_drop = quick_check([c for c in F_FEATS if c not in NOISE],
                     [(c, m) for c, m in SPEC_F if c not in NOISE], "F without non-recipe columns")

applied = []
if s_worry > 0.94586 + MIN_GAIN:
    BASE_FEATS += [c for c in WORRY if c not in BASE_FEATS]
    applied.append("expected-worry features")
if s_drop > 0.94586 + MIN_GAIN:
    BASE_FEATS[:] = [c for c in BASE_FEATS if c not in NOISE]
    SPEC_G[:] = [(c, m) for c, m in SPEC_G if c not in NOISE]
    applied.append("dropped non-recipe columns")
print(f"\nApplied for Cell 2: {applied if applied else 'nothing'}")
print(f"Cell 2 will use {len(BASE_FEATS)} base features and {len(SPEC_G)} encodings")


In [ ]:
# ===========================================================================
# CELL 2 - Full pipeline, every improvement included:
#   * features: raw + income/commute digits + value frequencies
#               + original-dataset features (frequency, lift, novelty,
#                 distance to nearest original value) if the dataset is attached
#   * nested target encoding on every column at two smoothings, income/commute
#     at three smoothings, plus multi-resolution income/commute encodings
#   * 10 folds (set N_FOLDS = 20 for the full version, ~2x runtime)
#   * encoding-seed bags: each seed reshuffles the inner encoding folds
#   * LightGBM: shallow, column-subsampled trees (depth 5, 30% of columns), lr 0.02
#   * CatBoost (GPU) and a PyTorch MLP on the same encoded features
#   * logit-space blend, weights chosen on out-of-fold predictions
# Runtime ~2.5-3 h on a GPU session: run via "Save Version -> Save & Run All".
# ===========================================================================
N_FOLDS = 10
SEEDS = [42, 7, 2026]
RUN_CATBOOST = True     # runs on the first seed only
RUN_MLP = True
USE_GPU_CATBOOST = torch.cuda.is_available()

FEATS = BASE_FEATS + ORIG_FEATS
SPEC = SPEC_G
LGB_PARAMS = dict(learning_rate=0.02, max_depth=5, num_leaves=31, min_child_samples=100,
                  reg_lambda=1.0, subsample=0.8, subsample_freq=1, colsample_bytree=0.3,
                  n_estimators=30000)
FOLDS = make_folds(N_FOLDS)

print(f"{len(FEATS)} features ({len(ORIG_FEATS)} from original data) | {len(SPEC)} encodings | "
      f"{N_FOLDS} folds | seeds {SEEDS} | CatBoost {RUN_CATBOOST} (GPU {USE_GPU_CATBOOST}) | "
      f"MLP {RUN_MLP} ({DEVICE})\n")

oof = {"lgb": [], "mlp": [], "cb": []}
tst = {"lgb": [], "mlp": [], "cb": []}
t_all = time.time()

for s_i, seed in enumerate(SEEDS):
    do_cb = RUN_CATBOOST and s_i == 0
    o_lgb, t_lgb = np.zeros(len(X_all)), np.zeros(len(X_test_all))
    o_mlp, t_mlp = np.zeros(len(X_all)), np.zeros(len(X_test_all))
    o_cb, t_cb = np.zeros(len(X_all)), np.zeros(len(X_test_all))
    t_seed = time.time()
    print(f"=== Seed {seed} ===")
    for fold, (tr, va) in enumerate(FOLDS):
        X_fit, X_val, X_tst = fold_data(tr, va, FEATS, SPEC, seed)
        msg = f"  fold {fold:2d}:"

        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed, verbose=-1)
        m.fit(X_fit, y[tr], eval_set=[(X_val, y[va])], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
        o_lgb[va] = m.predict_proba(X_val)[:, 1]
        t_lgb += m.predict_proba(X_tst)[:, 1] / N_FOLDS
        msg += f" LGB {roc_auc_score(y[va], o_lgb[va]):.5f} ({m.best_iteration_}t)"

        if RUN_MLP:
            A_fit, A_val, A_tst = mlp_matrices(X_fit, X_val, X_tst)
            pv, pt, ep = train_mlp(A_fit, y[tr], A_val, y[va], A_tst, seed)
            o_mlp[va] = pv
            t_mlp += pt / N_FOLDS
            msg += f" | MLP {roc_auc_score(y[va], pv):.5f} ({ep}ep)"

        if do_cb:
            from catboost import CatBoostClassifier
            cb_cats = [c for c in cat_cols if c in X_fit.columns]
            Xc = [X.copy() for X in (X_fit, X_val, X_tst)]
            for X in Xc:
                for c in cb_cats:
                    X[c] = X[c].astype(str)
            cb = CatBoostClassifier(
                iterations=10000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
                eval_metric="AUC", od_type="Iter", od_wait=200, random_seed=seed, verbose=0,
                task_type="GPU" if USE_GPU_CATBOOST else "CPU")
            cb.fit(Xc[0], y[tr], cat_features=cb_cats, eval_set=(Xc[1], y[va]), use_best_model=True)
            o_cb[va] = cb.predict_proba(Xc[1])[:, 1]
            t_cb += cb.predict_proba(Xc[2])[:, 1] / N_FOLDS
            msg += f" | CB {roc_auc_score(y[va], o_cb[va]):.5f}"
        print(msg)

    oof["lgb"].append(o_lgb); tst["lgb"].append(t_lgb)
    np.save(f"oof_lgb_{seed}.npy", o_lgb); np.save(f"test_lgb_{seed}.npy", t_lgb)
    line = f"  seed {seed}: LGB {roc_auc_score(y, o_lgb):.5f}"
    if RUN_MLP:
        oof["mlp"].append(o_mlp); tst["mlp"].append(t_mlp)
        np.save(f"oof_mlp_{seed}.npy", o_mlp); np.save(f"test_mlp_{seed}.npy", t_mlp)
        line += f" | MLP {roc_auc_score(y, o_mlp):.5f}"
    if do_cb:
        oof["cb"].append(o_cb); tst["cb"].append(t_cb)
        np.save("oof_cb.npy", o_cb); np.save("test_cb.npy", t_cb)
        line += f" | CB {roc_auc_score(y, o_cb):.5f}"
    print(line + f"  [{(time.time() - t_seed) / 60:.1f} min]\n")

# --- Average seeds per model ------------------------------------------------------
models = {k: (np.mean(oof[k], axis=0), np.mean(tst[k], axis=0)) for k in oof if oof[k]}
print("Seed-averaged OOF AUC per model:")
for k, (o, _) in models.items():
    print(f"  {k:4s} {roc_auc_score(y, o):.5f}")

# --- Logit-space blend: search the weight simplex (step 0.05) on OOF -----------------
names = list(models)
L_oof = {k: logit(models[k][0]) for k in names}
steps = np.round(np.arange(0, 1.0001, 0.05), 2)
best_w, best_auc = None, -1.0
for head in itertools.product(steps, repeat=len(names) - 1):
    last = round(1 - sum(head), 2)
    if last < 0:
        continue
    w = dict(zip(names, list(head) + [last]))
    score = roc_auc_score(y, sum(w[k] * L_oof[k] for k in names))
    if score > best_auc:
        best_w, best_auc = w, score

z = sum(best_w[k] * logit(models[k][1]) for k in names)
final_pred = 1 / (1 + np.exp(-z))

print("\n" + "=" * 60)
print("Blend weights:  " + ", ".join(f"{k} {best_w[k]:.2f}" for k in names))
print(f"Blend OOF AUC:  {best_auc:.5f}   (previous best: CV 0.94597, LB 0.94609)")
print(f"Total runtime:  {(time.time() - t_all) / 60:.1f} min")
save_submission(final_pred, "submission.csv")
